# ETL — Preço Real e Índice Encadeado por Item Pareado
**TCC — Preços e concentração de mercado em compras públicas de insumos hospitalares, 2009–2023**  
Mônica Anatalia Bezerra de Araujo — MBA em Data Science e Analytics para Operações, POLI USP PRO

**O que este notebook faz:**
1. Lê os parquets harmonizados e o deflator do IPCA
2. Deflaciona cada registro pelo **mês da compra** (base: dezembro/2023)
3. Calcula a série de preço mediano nominal e real por ano
4. Calcula o **índice encadeado por item pareado** (2009 = 100)
5. Executa a **análise de robustez** restringindo a itens de presença contínua
6. Exporta as tabelas para o Power BI

**Por que o índice pareado:** a mediana anual compara conjuntos diferentes de
produtos a cada ano; parte da variação seria mudança de composição da cesta e
não de preço. O índice pareado compara cada item consigo mesmo entre anos
consecutivos e encadeia as variações — mesma lógica de números-índice usada
pelo IBGE no IPCA.

## 1. Montar o Drive e instalar dependências

In [ ]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
# Ajuste PASTA_DADOS para o local onde estao os arquivos.
# Padrao: subpasta "dados" ao lado do notebook. No Google Colab, aponte para
# a pasta do seu Drive apos monta-lo.
import os
PASTA_DADOS = os.environ.get('BPS_DADOS', 'dados')
# ─────────────────────────────────────────────────────────────────────────
!pip install polars pyarrow --quiet
print('Pronto ✓')

## 2. Configuração
> ⚙️ Ajuste os caminhos. O recorte 2009–2023 é aplicado aqui.

In [ ]:
import polars as pl
from pathlib import Path

# ── AJUSTE AQUI ──────────────────────────────────────────────────────────
BASE     = Path(PASTA_DADOS)
PASTA_BPS   = BASE / 'Base Harmonizacao Campos'
ARQ_IPCA    = BASE / 'IPCA Tratado' / 'ipca_deflator.parquet'
PASTA_SAIDA = BASE / 'Base Preco Real'
ANO_INICIO, ANO_FIM = 2009, 2023      # recorte da analise
# ─────────────────────────────────────────────────────────────────────────

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
print(f'BPS   : {PASTA_BPS}')
print(f'IPCA  : {ARQ_IPCA}')
print(f'Saida : {PASTA_SAIDA}')
print(f'Recorte: {ANO_INICIO}-{ANO_FIM}')

## 3. Carregar e deflacionar
> O ano vem de `data_compra`, nao do nome do arquivo — arquivos anuais podem
> conter compras de anos anteriores inseridas com atraso no sistema.

In [ ]:
COLS = ['data_compra','preco_unitario','quantidade','valor_total',
        'cod_catmat','unidade_chave','uf','modalidade_compra']

frames = []
for ano in range(2000, 2026):
    arq = PASTA_BPS / f'BPS_{ano}.parquet'
    if not arq.exists():
        continue
    disp = pl.read_parquet(arq, n_rows=0).columns
    frames.append(pl.read_parquet(arq, columns=[c for c in COLS if c in disp]))
bruto = pl.concat(frames, how='diagonal')
print(f'Registros lidos: {bruto.height:,}')

ipca = pl.read_parquet(ARQ_IPCA).select(['ano','mes','fator_deflacao'])

bps = (bruto
    .filter(pl.col('data_compra').is_not_null()
            & pl.col('data_compra').dt.year().is_between(ANO_INICIO, ANO_FIM)
            & (pl.col('preco_unitario') > 0)
            & pl.col('cod_catmat').is_not_null())
    .with_columns(pl.col('data_compra').dt.year().alias('ano'),
                  pl.col('data_compra').dt.month().alias('mes'))
    .join(ipca, on=['ano','mes'], how='left')
    .with_columns((pl.col('preco_unitario') * pl.col('fator_deflacao')).alias('preco_real')))

sem_fator = bps.filter(pl.col('fator_deflacao').is_null()).height
print(f'Registros na analise : {bps.height:,}')
print(f'Sem fator de deflacao: {sem_fator:,}  (deve ser 0)')
assert sem_fator == 0, 'Ha meses sem cobertura no deflator do IPCA'

## 4. Série anual — mediana nominal e real

In [ ]:
serie = (bps.group_by('ano').agg(
            pl.len().alias('registros'),
            pl.col('cod_catmat').n_unique().alias('itens_distintos'),
            pl.col('preco_unitario').median().alias('mediana_nominal'),
            pl.col('preco_real').median().alias('mediana_real'))
          .sort('ano'))
base = serie.filter(pl.col('ano') == ANO_INICIO)['mediana_real'][0]
serie = serie.with_columns((pl.col('mediana_real')/base*100).round(1).alias('indice_mediana'))
print(serie)

## 5. Índice encadeado por item pareado
> Para cada par de anos consecutivos, usa-se apenas os itens presentes nos DOIS
> anos. Calcula-se a razao de preco real de cada item e toma-se a MEDIANA das
> razoes — assim cada item pesa igual, independentemente do valor. As razoes
> sucessivas sao MULTIPLICADAS (encadeadas), nao somadas.

In [ ]:
def indice_pareado(df, min_anos=None):
    """Indice encadeado 2009=100. min_anos restringe a itens presentes em
    pelo menos N anos distintos (usado na analise de robustez)."""
    d = (df.filter(pl.col('unidade_chave').is_not_null()
                   & (pl.col('unidade_chave').str.len_chars() > 0))
           .with_columns((pl.col('cod_catmat') + '|' + pl.col('unidade_chave')).alias('item')))
    if min_anos:
        persist = (d.group_by('item').agg(pl.col('ano').n_unique().alias('n'))
                     .filter(pl.col('n') >= min_anos).select('item'))
        d = d.join(persist, on='item', how='semi')

    med = d.group_by(['ano','item']).agg(pl.col('preco_real').median().alias('p'))
    idx, linhas = 100.0, [{'ano': ANO_INICIO, 'indice': 100.0,
                           'itens_no_elo': None, 'variacao_pct': None}]
    for a in range(ANO_INICIO, ANO_FIM):
        x = med.filter(pl.col('ano') == a).select(['item','p']).rename({'p':'p0'})
        y = med.filter(pl.col('ano') == a+1).select(['item','p']).rename({'p':'p1'})
        j = x.join(y, on='item', how='inner')
        if j.height == 0:
            linhas.append({'ano': a+1, 'indice': None, 'itens_no_elo': 0, 'variacao_pct': None})
            continue
        razao = (j['p1'] / j['p0']).median()
        idx *= razao
        linhas.append({'ano': a+1, 'indice': round(idx, 1),
                       'itens_no_elo': j.height, 'variacao_pct': round((razao-1)*100, 2)})
    return pl.DataFrame(linhas)

idx_publicado = indice_pareado(bps)
print(idx_publicado)

## 6. Análise de robustez
> Recalcula o indice restringindo a itens de presenca continua. Se o resultado
> se mantiver, a queda nao e artefato da rotatividade de produtos na base.

In [ ]:
cenarios = {'todos os itens': None, '>= 5 anos': 5, '>= 10 anos': 10, 'todos os 15 anos': 15}
resumo, detalhe = [], {}
for rot, minimo in cenarios.items():
    r = indice_pareado(bps, minimo)
    detalhe[rot] = r
    final = r.filter(pl.col('ano') == ANO_FIM)['indice'][0]
    n_itens = r['itens_no_elo'].max()
    resumo.append({'cenario': rot, 'indice_2023': final, 'max_itens_no_elo': n_itens})
robustez = pl.DataFrame(resumo)
print(robustez)
amp = robustez['indice_2023'].max() - robustez['indice_2023'].min()
print(f'\nAmplitude entre cenarios: {amp:.1f} pontos')

## 6b. Memória de cálculo do encadeamento — elo a elo
> Imprime a aritmetica completa de cada cenario: quantos itens entraram em cada
> elo, a razao mediana daquele elo e a multiplicacao que produz o indice.
> O objetivo e permitir que qualquer valor publicado seja refeito a mao.
>
> **Leitura:** razao acima de 1 indica alta de preco real no elo; abaixo de 1,
> queda. O indice do ano e o indice do ano anterior MULTIPLICADO pela razao —
> as variacoes se encadeiam, nao se somam.

In [ ]:
def memoria_calculo(df, min_anos=None, rotulo='todos os itens'):
    """Imprime a memoria de calculo do indice encadeado e devolve a tabela."""
    d = (df.filter(pl.col('unidade_chave').is_not_null()
                   & (pl.col('unidade_chave').str.len_chars() > 0))
           .with_columns((pl.col('cod_catmat') + '|' + pl.col('unidade_chave')).alias('item')))
    if min_anos:
        persist = (d.group_by('item').agg(pl.col('ano').n_unique().alias('k'))
                     .filter(pl.col('k') >= min_anos).select('item'))
        d = d.join(persist, on='item', how='semi')
    med = d.group_by(['ano','item']).agg(pl.col('preco_real').median().alias('p'))

    print(f'\n=== CENARIO: {rotulo} ===')
    print(f'{"elo":>11} | {"itens":>6} | {"razao mediana":>14} | {"conta":<32} | {"indice":>7}')
    idx, linhas = 100.0, []
    for a in range(ANO_INICIO, ANO_FIM):
        x = med.filter(pl.col('ano')==a).select(['item','p']).rename({'p':'p0'})
        y = med.filter(pl.col('ano')==a+1).select(['item','p']).rename({'p':'p1'})
        j = x.join(y, on='item', how='inner')
        if j.height == 0: continue
        razao = float((j['p1']/j['p0']).median())
        ant = idx; idx *= razao
        print(f'{a}\u2192{a+1:>4} | {j.height:>6,} | {razao:>14.6f} | '
              f'{ant:>7.2f} x {razao:.6f} = {idx:>7.2f} | {idx:>7.1f}')
        linhas.append({'cenario': rotulo, 'elo': f'{a}-{a+1}', 'itens_no_elo': j.height,
                       'razao_mediana': round(razao,6), 'indice_anterior': round(ant,2),
                       'indice_resultante': round(idx,2)})
    print(f'{"":>11}   {"":>6}   {"":>14}   {"INDICE FINAL":<32} | {idx:>7.1f}')
    return pl.DataFrame(linhas)

memorias = pl.concat([memoria_calculo(bps, m, r) for r, m in
                      [('todos os itens', None), ('>= 5 anos', 5),
                       ('>= 10 anos', 10), ('todos os 15 anos', 15)]])
memorias.write_parquet(PASTA_SAIDA / 'memoria_calculo_indice.parquet')
memorias.write_csv(PASTA_SAIDA / 'memoria_calculo_indice.csv')
print('\nMemoria de calculo exportada: memoria_calculo_indice.csv')
print('Conferencia: cesta fixa de 311 itens deve terminar em 75,3;')
print('             o elo 2021-2022 deve ter razao 0,892646.')

## 7. Exportar para o Power BI

In [ ]:
serie.write_parquet(PASTA_SAIDA / 'serie_preco_anual.parquet')
idx_publicado.write_parquet(PASTA_SAIDA / 'indice_pareado.parquet')
robustez.write_parquet(PASTA_SAIDA / 'robustez_indice.parquet')
serie.write_csv(PASTA_SAIDA / 'serie_preco_anual.csv')
idx_publicado.write_csv(PASTA_SAIDA / 'indice_pareado.csv')
robustez.write_csv(PASTA_SAIDA / 'robustez_indice.csv')

print('=== EXPORTADO ===')
for f in sorted(PASTA_SAIDA.iterdir()):
    print(f'  {f.name}')
print(f'\nIndice 2023 (publicado): {idx_publicado.filter(pl.col("ano")==ANO_FIM)["indice"][0]}')
print('Valores de referencia para conferencia:')
print('  serie: 883.652 registros | mediana real 2009 = 2,2133 | 2023 = 2,0921 | indice 94,5')
print('  pareado: 74,8 | robustez: 74,8 / 75,8 / 77,4 / 75,3')